# 01_Final_Dataset_Preparation

**Thesis:** Analysis of Medical Scientific Engagement Text Using Natural Language Processing and Large Language Models

This notebook creates and validates the final biomedical literature corpus before NLP analysis.

**Important:** Keep `pubmed_medical_affairs_3layer_corpus.csv` unchanged as the RAW dataset. Run this notebook from top to bottom. Do not continue using the old repair notebook for the final dataset.


## Step 1 — Import libraries

**RUN THIS CELL.**

In [1]:
import pandas as pd
import numpy as np
import requests
import time
import re
import html
import xml.etree.ElementTree as ET

RAW_FILE = "pubmed_medical_affairs_3layer_corpus.csv"
FINAL_FILE = "pubmed_medical_affairs_3layer_corpus_cleaned.csv"

# Replace with your own email before the PubMed retrieval step.
NCBI_EMAIL = "samitkumardas321@gmail.com"

BATCH_SIZE = 200
REQUEST_TIMEOUT = 60
MAX_RETRIES = 3

print("Libraries loaded.")


Libraries loaded.


## Step 2 — Load the RAW dataset

**RUN THIS CELL.**

Never overwrite the raw CSV.

In [2]:
df_raw = pd.read_csv(RAW_FILE)

print("Raw dataset shape:", df_raw.shape)
print("Columns:", df_raw.columns.tolist())
display(df_raw.head())


Raw dataset shape: (21256, 6)
Columns: ['pmid', 'title', 'abstract', 'year', 'journal', 'layer']


,pmid,title,abstract,year,journal,layer
0,41483736,Intracranial activity of antibody-drug conjuga...,Antibody-drug conjugates (ADCs) represent a ra...,2025.0,Cancer treatment reviews,L1_scientific_engagement
1,41483728,Effect of quetiapine versus haloperidol on del...,"Delirium is a neuropsychiatric condition, char...",2025.0,Asian journal of psychiatry,L1_scientific_engagement
2,41483727,Vision-language models in diagnostic imaging: ...,Radiology faces an unprecedented workload cris...,2025.0,International journal of medical informatics,L1_scientific_engagement
3,41483666,The complexity of dystrophin transcription and...,Duchenne muscular dystrophy (DMD) is a rare ge...,2025.0,Neuromuscular disorders : NMD,L1_scientific_engagement
4,41483580,EEG biomarkers for a precision-medicine approa...,Major depressive disorder (MDD) is a prevalent...,2025.0,Psychiatry research. Neuroimaging,L1_scientific_engagement


## Step 3 — Initial dataset audit

**RUN THIS CELL.**

In [3]:
df_raw.info()

print("\nMissing values:")
print(df_raw.isnull().sum())

print("\nMissing percentages:")
print((df_raw.isnull().sum() / len(df_raw) * 100).round(4))

print("\nCompletely duplicated rows:", df_raw.duplicated().sum())
print("Duplicated PMIDs:", df_raw["pmid"].duplicated().sum())
print("Duplicated titles:", df_raw["title"].duplicated().sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21256 entries, 0 to 21255
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   pmid      21256 non-null  int64  
 1   title     21214 non-null  object 
 2   abstract  21256 non-null  object 
 3   year      21248 non-null  float64
 4   journal   21256 non-null  object 
 5   layer     21256 non-null  object 
dtypes: float64(1), int64(1), object(4)
memory usage: 996.5+ KB

Missing values:
pmid         0
title       42
abstract     0
year         8
journal      0
layer        0
dtype: int64

Missing percentages:
pmid        0.0000
title       0.1976
abstract    0.0000
year        0.0376
journal     0.0000
layer       0.0000
dtype: float64

Completely duplicated rows: 0
Duplicated PMIDs: 0
Duplicated titles: 60


## Step 4 — Investigate repeated titles

Repeated titles are not automatically duplicate publications. Do not delete them solely because the title repeats.

In [4]:
duplicate_title_records = df_raw[df_raw["title"].duplicated(keep=False)].sort_values("title")

duplicate_title_abstract_count = duplicate_title_records.duplicated(
    subset=["title", "abstract"]
).sum()

print("Records with repeated titles:", len(duplicate_title_records))
print("Repeated exact title + abstract combinations:", duplicate_title_abstract_count)

display(duplicate_title_records[
    ["pmid", "title", "year", "journal", "layer"]
].head(20))


Records with repeated titles: 73
Repeated exact title + abstract combinations: 3


,pmid,title,year,journal,layer
17027,41384609,3D,2025.0,ACS biomaterials science & engineering,L3_comparative_critical
8396,41126959,3D,2025.0,Frontiers in pharmacology,L1_scientific_engagement
12884,41122842,Advanced Life Support: 2025 International Liai...,2025.0,Circulation,L2_safety_risk
12918,41117578,Advanced Life Support: 2025 International Liai...,2025.0,Resuscitation,L2_safety_risk
987,41433764,Alzheimer's Imaging Consortium.,2025.0,Alzheimer's & dementia : the journal of the Al...,L1_scientific_engagement
988,41433740,Alzheimer's Imaging Consortium.,2025.0,Alzheimer's & dementia : the journal of the Al...,L1_scientific_engagement
989,41433458,Alzheimer's Imaging Consortium.,2025.0,Alzheimer's & dementia : the journal of the Al...,L1_scientific_engagement
16228,41433513,Alzheimer's Imaging Consortium.,2025.0,Alzheimer's & dementia : the journal of the Al...,L3_comparative_critical
16227,41434179,Alzheimer's Imaging Consortium.,2025.0,Alzheimer's & dementia : the journal of the Al...,L3_comparative_critical
385,41464967,Application of,2025.0,"Foods (Basel, Switzerland)",L1_scientific_engagement


## Step 5 — Prepare PMIDs for PubMed verification

**RUN THIS CELL.**

In [5]:
df = df_raw.copy()
df["pmid"] = df["pmid"].astype(str).str.strip()

pmids = df["pmid"].tolist()

print("PMIDs prepared:", len(pmids))
print("Unique PMIDs:", df["pmid"].nunique())


PMIDs prepared: 21256
Unique PMIDs: 21256


## Step 6 — Retrieve abstracts from PubMed

**RUN THIS CELL.**

This is the longest-running step. With 21,256 PMIDs and a batch size of 200, about 107 requests are required. It may take several minutes. Progress is printed for every batch.

**Do not interrupt the kernel while it is running.**


In [6]:
def clean_abstract_text(text):
    if text is None:
        return ""
    text = html.unescape(str(text))
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_abstracts_from_xml(xml_text):
    abstracts = {}
    root = ET.fromstring(xml_text)

    for article in root.findall(".//PubmedArticle"):
        pmid_node = article.find(".//MedlineCitation/PMID")
        if pmid_node is None:
            continue

        pmid = str(pmid_node.text).strip()
        abstract_node = article.find(".//MedlineCitation/Article/Abstract")

        if abstract_node is None:
            abstracts[pmid] = ""
            continue

        parts = []
        for part in abstract_node.findall("AbstractText"):
            text = "".join(part.itertext()).strip()
            label = part.attrib.get("Label")

            if label and text:
                text = f"{label}: {text}"

            if text:
                parts.append(text)

        abstracts[pmid] = clean_abstract_text(" ".join(parts))

    return abstracts


url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

all_abstracts = {}
failed_batches = []

total_batches = (len(pmids) + BATCH_SIZE - 1) // BATCH_SIZE

print("Total PMIDs:", len(pmids))
print("Total batches:", total_batches)
print()

for batch_number, start in enumerate(
    range(0, len(pmids), BATCH_SIZE), start=1
):
    batch_pmids = pmids[start:start + BATCH_SIZE]

    params = {
        "db": "pubmed",
        "id": ",".join(batch_pmids),
        "retmode": "xml",
        "rettype": "abstract",
        "tool": "masters_thesis_nlp",
        "email": NCBI_EMAIL
    }

    success = False

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.get(
                url,
                params=params,
                timeout=REQUEST_TIMEOUT
            )

            if response.status_code == 200:
                all_abstracts.update(
                    extract_abstracts_from_xml(response.text)
                )
                success = True
                break

            print(
                f"Batch {batch_number}: HTTP {response.status_code} "
                f"(attempt {attempt}/{MAX_RETRIES})"
            )

        except Exception as e:
            print(
                f"Batch {batch_number}: {type(e).__name__}: {e} "
                f"(attempt {attempt}/{MAX_RETRIES})"
            )

        time.sleep(2)

    if not success:
        failed_batches.append(batch_number)

    time.sleep(0.4)

    print(
        f"Batch {batch_number}/{total_batches} | "
        f"Records: {len(batch_pmids)} | "
        f"Abstracts retrieved: {len(all_abstracts)}"
    )

print("\nRETRIEVAL COMPLETED")
print("===================")
print("Original PMIDs:", len(pmids))
print("Abstract records returned:", len(all_abstracts))
print("Failed batches:", failed_batches)


Total PMIDs: 21256
Total batches: 107

Batch 1/107 | Records: 200 | Abstracts retrieved: 200
Batch 2/107 | Records: 200 | Abstracts retrieved: 400
Batch 3/107 | Records: 200 | Abstracts retrieved: 600
Batch 4/107 | Records: 200 | Abstracts retrieved: 800
Batch 5/107 | Records: 200 | Abstracts retrieved: 997
Batch 6/107 | Records: 200 | Abstracts retrieved: 1197
Batch 7/107 | Records: 200 | Abstracts retrieved: 1397
Batch 8/107 | Records: 200 | Abstracts retrieved: 1596
Batch 9/107 | Records: 200 | Abstracts retrieved: 1796
Batch 10/107 | Records: 200 | Abstracts retrieved: 1996
Batch 11/107 | Records: 200 | Abstracts retrieved: 2196
Batch 12/107 | Records: 200 | Abstracts retrieved: 2396
Batch 13/107 | Records: 200 | Abstracts retrieved: 2596
Batch 14/107 | Records: 200 | Abstracts retrieved: 2796
Batch 15/107 | Records: 200 | Abstracts retrieved: 2996
Batch 16/107 | Records: 200 | Abstracts retrieved: 3196
Batch 17/107 | Records: 200 | Abstracts retrieved: 3396
Batch 18/107 | Records:

## Step 7 — Identify unmatched/empty PubMed results

**RUN THIS CELL.**

These records are retained; they are not deleted.

In [7]:
retrieved_pmids = set(all_abstracts.keys())

missing_pmids = sorted(set(pmids) - retrieved_pmids)

empty_retrieved_pmids = sorted(
    pmid for pmid, abstract in all_abstracts.items()
    if not str(abstract).strip()
)

print("Original PMIDs:", len(pmids))
print("PMIDs returned by PubMed:", len(retrieved_pmids))
print("PMIDs not returned by PubMed:", len(missing_pmids))
print("Returned PMIDs with empty abstracts:", len(empty_retrieved_pmids))

if missing_pmids:
    print("\nPMIDs not returned:")
    print(missing_pmids)

if empty_retrieved_pmids:
    print("\nReturned PMIDs with empty abstracts:")
    print(empty_retrieved_pmids)


Original PMIDs: 21256
PMIDs returned by PubMed: 21241
PMIDs not returned by PubMed: 15
Returned PMIDs with empty abstracts: 2

PMIDs not returned:
['41219071', '41412986', '41412990', '41433266', '41433272', '41433318', '41433361', '41433371', '41433458', '41433513', '41433740', '41433764', '41434179', '41468071', '41468091']

Returned PMIDs with empty abstracts:
['41372067', '41436046']


## Step 8 — Repair the abstract column

**RUN THIS CELL.**

Repair rule:

1. Non-empty PubMed abstract → use it.
2. PubMed does not return the PMID → retain original CSV abstract.
3. PubMed returns an empty abstract → retain original CSV abstract.
4. No publication is deleted.
5. Missing titles/years are not fabricated.


In [8]:
df_cleaned = df.copy()

df_cleaned["abstract_original"] = df_cleaned["abstract"]
df_cleaned["abstract_retrieved"] = df_cleaned["pmid"].map(all_abstracts)

valid_retrieved = (
    df_cleaned["abstract_retrieved"].notna()
    & df_cleaned["abstract_retrieved"].astype(str).str.strip().ne("")
)

df_cleaned.loc[valid_retrieved, "abstract"] = (
    df_cleaned.loc[valid_retrieved, "abstract_retrieved"]
)

print("Total records:", len(df_cleaned))
print("PubMed abstracts used:", int(valid_retrieved.sum()))
print("Records retaining original abstract:", int((~valid_retrieved).sum()))


Total records: 21256
PubMed abstracts used: 21239
Records retaining original abstract: 17


## Step 9 — Keep only the six thesis dataset attributes

**RUN THIS CELL.**

In [9]:
expected_columns = [
    "pmid", "title", "abstract", "year", "journal", "layer"
]

df_cleaned = df_cleaned.drop(
    columns=["abstract_original", "abstract_retrieved"]
)

df_cleaned = df_cleaned[expected_columns]

print("Cleaned dataset shape:", df_cleaned.shape)
print("Columns:", df_cleaned.columns.tolist())


Cleaned dataset shape: (21256, 6)
Columns: ['pmid', 'title', 'abstract', 'year', 'journal', 'layer']


## Step 10 — Final abstract quality checks

**RUN THIS CELL.**

In [10]:
df_cleaned["abstract"] = df_cleaned["abstract"].fillna("").astype(str)
df_cleaned["abstract_length"] = df_cleaned["abstract"].str.len()

empty_abstracts = df_cleaned["abstract"].str.strip().eq("").sum()
short_abstracts = (df_cleaned["abstract_length"] < 200).sum()

print("Total records:", len(df_cleaned))
print("Empty abstracts:", empty_abstracts)
print("Abstracts < 200 characters:", short_abstracts)

print("\nAbstract length statistics:")
print(df_cleaned["abstract_length"].describe())

print("\nShortest abstracts:")
display(
    df_cleaned.sort_values("abstract_length")[
        ["pmid", "title", "abstract_length", "layer", "abstract"]
    ].head(10)
)


Total records: 21256
Empty abstracts: 0
Abstracts < 200 characters: 1

Abstract length statistics:
count    21256.000000
mean      1723.961423
std        589.575058
min        127.000000
25%       1370.000000
50%       1680.000000
75%       1983.000000
max      14210.000000
Name: abstract_length, dtype: float64

Shortest abstracts:


,pmid,title,abstract_length,layer,abstract
16216,41436046,[The applications and challenges of rapid dete...,127,L3_comparative_critical,法医毒物学作为法医学的重要分支，是连接中毒事件调查、司法公正裁决与公共安全保障的关键技术支撑...
15647,41468091,Adjuvant Chemoradiotherapy Versus Radiotherapy...,313,L3_comparative_critical,(Abstracted from Lancet Oncol 2025;26(10):1370...
9004,41101910,Orthobiologics for Meniscus Injuries.,315,L1_scientific_engagement,The biologic enhancement of meniscal tears aim...
13341,41079276,Endoscopic strategies for the management of lo...,317,L2_safety_risk,Endoscopic resection is the standard approach ...
6943,41180001,Efficacy and Safety of Bictegravir Plus Lenaca...,322,L1_scientific_engagement,ARTISTRY-1 is a phase 2/3 trial of bictegravir...
1811,41392927,R WE ready for reimbursement? A round-up of de...,336,L1_scientific_engagement,"In this update, we explore a review on transpo..."
8778,41109726,Advantages and Disadvantages of the Arthroscop...,349,L1_scientific_engagement,Arthroscopic ankle fusion has become the prefe...
15182,40991743,Attaching the Sacrocolpopexy Mesh to the Sacrum.,359,L2_safety_risk,Attachment of the mesh to the sacral anterior ...
1886,41389712,TL1A blockade in inflammatory bowel diseases: ...,362,L1_scientific_engagement,TL1A inhibitors offer a novel therapeutic stra...
5566,41233099,Updates on Multimodal Analgesia for Orthopedic...,369,L1_scientific_engagement,Multimodal analgesia has emerged as the standa...


## Step 11 — Final metadata and duplicate validation

**RUN THIS CELL.**

In [11]:
print("Missing values:")
print(df_cleaned[expected_columns].isnull().sum())

print("\nDuplicate complete rows:")
print(df_cleaned.duplicated().sum())

print("\nDuplicate PMIDs:")
print(df_cleaned["pmid"].duplicated().sum())

print("\nDuplicate title + abstract combinations:")
print(df_cleaned.duplicated(subset=["title", "abstract"]).sum())

print("\nLayer distribution:")
print(df_cleaned["layer"].value_counts(dropna=False))


Missing values:
pmid         0
title       42
abstract     0
year         8
journal      0
layer        0
dtype: int64

Duplicate complete rows:
0

Duplicate PMIDs:
0

Duplicate title + abstract combinations:
3

Layer distribution:
layer
L1_scientific_engagement    9599
L3_comparative_critical     5849
L2_safety_risk              5808
Name: count, dtype: int64


## Step 12 — Inspect short abstracts

**RUN THIS CELL.**

Do not automatically delete short abstracts. A short abstract may be legitimate, including a non-English abstract. Inspect the records before making any exclusion decision.


In [12]:
short_records = df_cleaned[
    df_cleaned["abstract_length"] < 200
][[
    "pmid", "title", "year", "journal",
    "layer", "abstract_length", "abstract"
]]

display(short_records)


,pmid,title,year,journal,layer,abstract_length,abstract
16216,41436046,[The applications and challenges of rapid dete...,2025.0,Fa yi xue za zhi,L3_comparative_critical,127,法医毒物学作为法医学的重要分支，是连接中毒事件调查、司法公正裁决与公共安全保障的关键技术支撑...


In [17]:
df_final.columns

Index(['pmid', 'title', 'abstract', 'year', 'journal', 'layer'], dtype='object')

## Step 13 — Save the FINAL analysis dataset

**RUN THIS CELL only after the validation cells show the expected results.**


In [13]:
final_columns = [
    "pmid", "title", "abstract", "year", "journal", "layer"
]

df_final = df_cleaned[final_columns].copy()

df_final.to_csv(
    FINAL_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("FINAL DATASET SAVED")
print("===================")
print("File:", FINAL_FILE)
print("Shape:", df_final.shape)
print("Columns:", df_final.columns.tolist())


FINAL DATASET SAVED
File: pubmed_medical_affairs_3layer_corpus_cleaned.csv
Shape: (21256, 6)
Columns: ['pmid', 'title', 'abstract', 'year', 'journal', 'layer']


## Step 14 — Final freeze check

**RUN THIS CELL.**

If this passes, the dataset-preparation stage is finished. Do not make further manual changes to the CSV.

The next notebook will be:

**`02_EDA_and_Text_Analysis.ipynb`**


In [14]:
df_final_check = pd.read_csv(FINAL_FILE)

print("FINAL DATASET VERIFICATION")
print("==========================")
print("Shape:", df_final_check.shape)
print("Columns:", df_final_check.columns.tolist())

print("\nMissing values:")
print(df_final_check.isnull().sum())

print("\nDuplicate PMIDs:",
      df_final_check["pmid"].duplicated().sum())

print(
    "Empty abstracts:",
    df_final_check["abstract"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

assert df_final_check.shape[0] == 21256, (
    f"Expected 21,256 records, found {df_final_check.shape[0]}"
)

assert df_final_check["pmid"].duplicated().sum() == 0, (
    "Duplicate PMIDs detected."
)

assert (
    df_final_check["abstract"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
    == 0
), "Empty abstracts detected."

print("\nDATASET FREEZE CHECK PASSED.")
print("The cleaned CSV is ready for downstream NLP analysis.")


FINAL DATASET VERIFICATION
Shape: (21256, 6)
Columns: ['pmid', 'title', 'abstract', 'year', 'journal', 'layer']

Missing values:
pmid         0
title       42
abstract     0
year         8
journal      0
layer        0
dtype: int64

Duplicate PMIDs: 0
Empty abstracts: 0

DATASET FREEZE CHECK PASSED.
The cleaned CSV is ready for downstream NLP analysis.


In [1]:
import pandas as pd
df = pd.read_csv("pubmed_medical_affairs_3layer_corpus_cleaned.csv")

In [3]:
# Check publication year distribution
year_counts = df["year"].value_counts().sort_index()

print(year_counts)

year
2024.0       13
2025.0    19714
2026.0     1521
Name: count, dtype: int64


In [4]:
# Show records labelled as 2026
df_2026 = df[df["year"] == 2026]

print("Number of 2026 records:", len(df_2026))

df_2026[["pmid", "title", "year", "journal", "layer"]].head(20)

Number of 2026 records: 1521


,pmid,title,year,journal,layer
19,41481829,Lung-morphology-guided mechanical ventilation ...,2026.0,Current opinion in critical care,L1_scientific_engagement
28,41480483,Extracellular vesicles in viral disease manage...,2026.0,Current research in microbial sciences,L1_scientific_engagement
97,41477657,Tobacco cessation strategies in military perso...,2026.0,Preventive medicine reports,L1_scientific_engagement
113,41477216,Exosome-Enhanced Injectable Hydrogels: A Compr...,2026.0,Health science reports,L1_scientific_engagement
114,41477213,Safety and Efficacy of Nephron-Sparing Surgery...,2026.0,Health science reports,L1_scientific_engagement
126,41476954,Eye tracking as a diagnostic tool in Alzheimer...,2026.0,"Alzheimer's & dementia (Amsterdam, Netherlands)",L1_scientific_engagement
147,41475715,Chitosan and chito-oligosaccharides as multifu...,2026.0,Carbohydrate polymers,L1_scientific_engagement
165,41474966,T cell engagers emerge as a compelling therape...,2026.0,The Journal of experimental medicine,L1_scientific_engagement
175,41473843,The role of vitamin D receptor signaling in ha...,2026.0,Journal of cell communication and signaling,L1_scientific_engagement
179,41473679,Fluid biopsies for,2026.0,Medicine international,L1_scientific_engagement


In [5]:
# Display the PMIDs and titles of 2026 records
# so we can investigate their actual PubMed publication dates

df_2026[["pmid", "title", "year"]].head(50)

,pmid,title,year
19,41481829,Lung-morphology-guided mechanical ventilation ...,2026.0
28,41480483,Extracellular vesicles in viral disease manage...,2026.0
97,41477657,Tobacco cessation strategies in military perso...,2026.0
113,41477216,Exosome-Enhanced Injectable Hydrogels: A Compr...,2026.0
114,41477213,Safety and Efficacy of Nephron-Sparing Surgery...,2026.0
126,41476954,Eye tracking as a diagnostic tool in Alzheimer...,2026.0
147,41475715,Chitosan and chito-oligosaccharides as multifu...,2026.0
165,41474966,T cell engagers emerge as a compelling therape...,2026.0
175,41473843,The role of vitamin D receptor signaling in ha...,2026.0
179,41473679,Fluid biopsies for,2026.0


In [6]:
print("Minimum year:", df["year"].min())
print("Maximum year:", df["year"].max())

print("\nYear distribution:")
print(df["year"].value_counts().sort_index())

Minimum year: 2024.0
Maximum year: 2026.0

Year distribution:
year
2024.0       13
2025.0    19714
2026.0     1521
Name: count, dtype: int64


In [2]:
import requests
from Bio import Entrez

Entrez.email = "samitkumardas321@gmail.com"

pmid = "41481829"

handle = Entrez.efetch(
    db="pubmed",
    id=pmid,
    retmode="xml"
)

record = Entrez.read(handle)

article = record["PubmedArticle"][0]
medline = article["MedlineCitation"]
pubmed_article = medline["Article"]

print("PMID:", pmid)

print("\n--- JOURNAL PUBLICATION DATE ---")
print(pubmed_article["Journal"]["JournalIssue"]["PubDate"])

print("\n--- ARTICLE DATE ---")
print(pubmed_article.get("ArticleDate"))

print("\n--- PUBMED HISTORY ---")
print(article.get("PubmedData", {}).get("History"))

PMID: 41481829

--- JOURNAL PUBLICATION DATE ---
{'Year': '2026', 'Month': 'Feb', 'Day': '01'}

--- ARTICLE DATE ---
[DictElement({'Year': '2025', 'Month': '11', 'Day': '27'}, attributes={'DateType': 'Electronic'})]

--- PUBMED HISTORY ---
[DictElement({'Year': '2026', 'Month': '1', 'Day': '2', 'Hour': '19', 'Minute': '39'}, attributes={'PubStatus': 'medline'}), DictElement({'Year': '2026', 'Month': '1', 'Day': '2', 'Hour': '19', 'Minute': '38'}, attributes={'PubStatus': 'pubmed'}), DictElement({'Year': '2026', 'Month': '1', 'Day': '2', 'Hour': '15', 'Minute': '13'}, attributes={'PubStatus': 'entrez'})]


In [3]:
from Bio import Entrez

Entrez.email = "YOUR_EMAIL"

# Take 10 examples currently labelled as 2026
sample_pmids = (
    df.loc[df["year"] == 2026, "pmid"]
      .astype(str)
      .head(10)
      .tolist()
)

for pmid in sample_pmids:

    handle = Entrez.efetch(
        db="pubmed",
        id=pmid,
        retmode="xml"
    )

    record = Entrez.read(handle)

    article = record["PubmedArticle"][0]
    medline = article["MedlineCitation"]
    art = medline["Article"]

    print("\n" + "="*60)
    print("PMID:", pmid)
    print("Dataset year:", 2026)

    print("Journal PubDate:",
          art["Journal"]["JournalIssue"].get("PubDate"))

    print("ArticleDate:",
          art.get("ArticleDate"))

    print("="*60)


PMID: 41481829
Dataset year: 2026
Journal PubDate: {'Year': '2026', 'Month': 'Feb', 'Day': '01'}
ArticleDate: [DictElement({'Year': '2025', 'Month': '11', 'Day': '27'}, attributes={'DateType': 'Electronic'})]

PMID: 41480483
Dataset year: 2026
Journal PubDate: {'Year': '2026'}
ArticleDate: [DictElement({'Year': '2025', 'Month': '12', 'Day': '04'}, attributes={'DateType': 'Electronic'})]

PMID: 41477657
Dataset year: 2026
Journal PubDate: {'Year': '2026', 'Month': 'Jan'}
ArticleDate: [DictElement({'Year': '2025', 'Month': '12', 'Day': '01'}, attributes={'DateType': 'Electronic'})]

PMID: 41477216
Dataset year: 2026
Journal PubDate: {'Year': '2026', 'Month': 'Jan'}
ArticleDate: [DictElement({'Year': '2025', 'Month': '12', 'Day': '29'}, attributes={'DateType': 'Electronic'})]

PMID: 41477213
Dataset year: 2026
Journal PubDate: {'Year': '2026', 'Month': 'Jan'}
ArticleDate: [DictElement({'Year': '2025', 'Month': '12', 'Day': '29'}, attributes={'DateType': 'Electronic'})]

PMID: 41476954
Da

In [5]:
from Bio import Entrez
import time

Entrez.email = "samitkumardas321@gmail.com"

# All records currently labelled 2026
pmids_2026 = (
    df.loc[df["year"] == 2026, "pmid"]
      .astype(str)
      .tolist()
)

print("Number of 2026 records:", len(pmids_2026))

Number of 2026 records: 1521


In [6]:
def extract_pub_dates(pmids):

    handle = Entrez.efetch(
        db="pubmed",
        id=",".join(pmids),
        retmode="xml"
    )

    record = Entrez.read(handle)

    results = []

    for article in record["PubmedArticle"]:

        medline = article["MedlineCitation"]
        pmid = str(medline["PMID"])

        art = medline["Article"]

        # Journal issue publication date
        journal_pubdate = (
            art["Journal"]["JournalIssue"].get("PubDate", {})
        )

        journal_year = journal_pubdate.get("Year", "")

        # Electronic article date
        article_dates = art.get("ArticleDate", [])

        electronic_year = ""

        for date in article_dates:
            if date.attributes.get("DateType") == "Electronic":
                electronic_year = date.get("Year", "")
                break

        results.append({
            "pmid": pmid,
            "journal_issue_year": journal_year,
            "electronic_year": electronic_year
        })

    return results

In [7]:
all_date_results = []

batch_size = 200

for i in range(0, len(pmids_2026), batch_size):

    batch = pmids_2026[i:i + batch_size]

    print(
        f"Processing batch {i//batch_size + 1} "
        f"of {(len(pmids_2026) + batch_size - 1)//batch_size}"
    )

    results = extract_pub_dates(batch)

    all_date_results.extend(results)

    time.sleep(0.3)

Processing batch 1 of 8
Processing batch 2 of 8
Processing batch 3 of 8
Processing batch 4 of 8
Processing batch 5 of 8
Processing batch 6 of 8
Processing batch 7 of 8
Processing batch 8 of 8


In [8]:
date_check = pd.DataFrame(all_date_results)

date_check.head()

,pmid,journal_issue_year,electronic_year
0,41481829,2026,2025
1,41480483,2026,2025
2,41477657,2026,2025
3,41477216,2026,2025
4,41477213,2026,2025


In [9]:
pd.crosstab(
    date_check["journal_issue_year"],
    date_check["electronic_year"]
)

electronic_year,2024,2025
journal_issue_year,,
2026,1,1520


In [10]:
date_check[
    (date_check["journal_issue_year"] == "2026") &
    (date_check["electronic_year"] == "2024")
]

,pmid,journal_issue_year,electronic_year
1229,41312874,2026,2024


In [12]:
outlier_pmid = date_check.loc[
    (date_check["journal_issue_year"] == "2026") &
    (date_check["electronic_year"] == "2024"),
    "pmid"
].iloc[0]

df[df["pmid"].astype(str) == str(outlier_pmid)]

,pmid,title,abstract,year,journal,layer
18347,41312874,Emotion Regulation in the Association Between ...,Posttraumatic stress disorder (PTSD) and subst...,2026.0,"Trauma, violence & abuse",L3_comparative_critical


In [13]:
date_check[
    (date_check["journal_issue_year"] == "2026") &
    (date_check["electronic_year"] == "2024")
]

,pmid,journal_issue_year,electronic_year
1229,41312874,2026,2024


In [14]:
df["journal_issue_year"] = df["year"]

In [16]:
# Ensure year column is numeric
df["year"] = pd.to_numeric(df["year"], errors="coerce")

# Create PMID → electronic publication year mapping
electronic_year_map = dict(
    zip(
        date_check["pmid"].astype(str),
        pd.to_numeric(date_check["electronic_year"], errors="coerce")
    )
)

# Find records currently classified as 2026
mask_2026 = df["year"] == 2026

# Replace 2026 with electronic publication year
df.loc[mask_2026, "year"] = (
    df.loc[mask_2026, "pmid"]
      .astype(str)
      .map(electronic_year_map)
)

In [17]:
print(df["year"].dtype)
print(df["year"].value_counts().sort_index())

float64
year
2024.0       14
2025.0    21234
Name: count, dtype: int64


In [18]:
df[df["year"] == 2026][["pmid", "year"]].head(20)

,pmid,year


In [19]:
df["year"].max()

2025.0

In [20]:
print("Dataset shape:", df.shape)
print("Unique PMIDs:", df["pmid"].nunique())

Dataset shape: (21256, 7)
Unique PMIDs: 21256


In [22]:
from Bio import Entrez
import pandas as pd
import time

Entrez.email = "samitkumardas321@gmail.com"

all_pmids = df["pmid"].astype(str).tolist()

def extract_publication_dates(pmids):

    handle = Entrez.efetch(
        db="pubmed",
        id=",".join(pmids),
        retmode="xml"
    )

    record = Entrez.read(handle)

    results = []

    for article in record["PubmedArticle"]:

        medline = article["MedlineCitation"]
        pmid = str(medline["PMID"])
        art = medline["Article"]

        # Journal issue publication date
        journal_pubdate = art["Journal"]["JournalIssue"].get("PubDate", {})

        journal_year = journal_pubdate.get("Year", "")
        journal_month = journal_pubdate.get("Month", "")
        journal_day = journal_pubdate.get("Day", "")
        journal_season = journal_pubdate.get("Season", "")

        # ArticleDate — usually electronic article date
        article_dates = art.get("ArticleDate", [])

        electronic_year = ""
        electronic_month = ""
        electronic_day = ""

        for date in article_dates:

            date_type = date.attributes.get("DateType")

            if date_type == "Electronic":

                electronic_year = date.get("Year", "")
                electronic_month = date.get("Month", "")
                electronic_day = date.get("Day", "")

                break

        # Publication history
        history = article.get("PubmedData", {}).get("History", [])

        history_dates = {}

        for h in history:

            status = h.attributes.get("PubStatus")

            if status:
                history_dates[status] = {
                    "year": h.get("Year", ""),
                    "month": h.get("Month", ""),
                    "day": h.get("Day", "")
                }

        results.append({
            "pmid": pmid,

            "journal_year": journal_year,
            "journal_month": journal_month,
            "journal_day": journal_day,
            "journal_season": journal_season,

            "electronic_year": electronic_year,
            "electronic_month": electronic_month,
            "electronic_day": electronic_day,

            "history_epublish_year":
                history_dates.get("epublish", {}).get("year", ""),

            "history_aheadofprint_year":
                history_dates.get("aheadofprint", {}).get("year", ""),

            "history_ppublish_year":
                history_dates.get("ppublish", {}).get("year", ""),

            "history_ecollection_year":
                history_dates.get("ecollection", {}).get("year", "")
        })

    return results

In [23]:
all_date_results = []

batch_size = 200

total_batches = (
    len(all_pmids) + batch_size - 1
) // batch_size

for i in range(0, len(all_pmids), batch_size):

    batch = all_pmids[i:i + batch_size]

    print(
        f"Processing batch "
        f"{i // batch_size + 1} of {total_batches}"
    )

    try:

        results = extract_publication_dates(batch)

        all_date_results.extend(results)

    except Exception as e:

        print(
            f"ERROR in batch "
            f"{i // batch_size + 1}: {e}"
        )

    time.sleep(0.3)

Processing batch 1 of 107
Processing batch 2 of 107
Processing batch 3 of 107
Processing batch 4 of 107
Processing batch 5 of 107
Processing batch 6 of 107
Processing batch 7 of 107
Processing batch 8 of 107
Processing batch 9 of 107
Processing batch 10 of 107
Processing batch 11 of 107
Processing batch 12 of 107
Processing batch 13 of 107
Processing batch 14 of 107
Processing batch 15 of 107
Processing batch 16 of 107
Processing batch 17 of 107
Processing batch 18 of 107
Processing batch 19 of 107
Processing batch 20 of 107
Processing batch 21 of 107
Processing batch 22 of 107
Processing batch 23 of 107
Processing batch 24 of 107
Processing batch 25 of 107
Processing batch 26 of 107
Processing batch 27 of 107
Processing batch 28 of 107
Processing batch 29 of 107
Processing batch 30 of 107
Processing batch 31 of 107
Processing batch 32 of 107
Processing batch 33 of 107
Processing batch 34 of 107
Processing batch 35 of 107
Processing batch 36 of 107
Processing batch 37 of 107
Processing

In [24]:
all_date_results = []

batch_size = 200

total_batches = (
    len(all_pmids) + batch_size - 1
) // batch_size

for i in range(0, len(all_pmids), batch_size):

    batch = all_pmids[i:i + batch_size]

    print(
        f"Processing batch "
        f"{i // batch_size + 1} of {total_batches}"
    )

    try:

        results = extract_publication_dates(batch)

        all_date_results.extend(results)

    except Exception as e:

        print(
            f"ERROR in batch "
            f"{i // batch_size + 1}: {e}"
        )

    time.sleep(0.3)

Processing batch 1 of 107
Processing batch 2 of 107
Processing batch 3 of 107
Processing batch 4 of 107
Processing batch 5 of 107
Processing batch 6 of 107
Processing batch 7 of 107
Processing batch 8 of 107
Processing batch 9 of 107
Processing batch 10 of 107
Processing batch 11 of 107
Processing batch 12 of 107
Processing batch 13 of 107
Processing batch 14 of 107
Processing batch 15 of 107
Processing batch 16 of 107
Processing batch 17 of 107
Processing batch 18 of 107
Processing batch 19 of 107
Processing batch 20 of 107
Processing batch 21 of 107
Processing batch 22 of 107
Processing batch 23 of 107
Processing batch 24 of 107
Processing batch 25 of 107
Processing batch 26 of 107
Processing batch 27 of 107
Processing batch 28 of 107
Processing batch 29 of 107
Processing batch 30 of 107
Processing batch 31 of 107
Processing batch 32 of 107
Processing batch 33 of 107
Processing batch 34 of 107
Processing batch 35 of 107
Processing batch 36 of 107
Processing batch 37 of 107
Processing

In [25]:
date_check_all = pd.DataFrame(all_date_results)

print("Records retrieved:", len(date_check_all))
print("Unique PMIDs:", date_check_all["pmid"].nunique())

Records retrieved: 21241
Unique PMIDs: 21241


In [26]:
# Find the 15 PMIDs missing from the date retrieval
retrieved_pmids = set(
    date_check_all["pmid"].astype(str)
)

all_pmids_set = set(
    df["pmid"].astype(str)
)

missing_pmids = sorted(
    all_pmids_set - retrieved_pmids
)

print("Missing PMIDs:", len(missing_pmids))
print(missing_pmids)

Missing PMIDs: 15
['41219071', '41412986', '41412990', '41433266', '41433272', '41433318', '41433361', '41433371', '41433458', '41433513', '41433740', '41433764', '41434179', '41468071', '41468091']


In [27]:
# Check whether the missing PMIDs are among the 15
# previously identified as having no retrievable PubMed abstract

print(missing_pmids)

['41219071', '41412986', '41412990', '41433266', '41433272', '41433318', '41433361', '41433371', '41433458', '41433513', '41433740', '41433764', '41434179', '41468071', '41468091']


In [28]:
from Bio import Entrez
import time

Entrez.email = "YOUR_EMAIL"

missing_date_results = []

for pmid in missing_pmids:

    try:
        handle = Entrez.efetch(
            db="pubmed",
            id=pmid,
            retmode="xml"
        )

        record = Entrez.read(handle)

        if len(record["PubmedArticle"]) == 0:
            print("NO RECORD:", pmid)
            continue

        article = record["PubmedArticle"][0]
        medline = article["MedlineCitation"]
        art = medline["Article"]

        journal_pubdate = (
            art["Journal"]["JournalIssue"].get("PubDate", {})
        )

        journal_year = journal_pubdate.get("Year", "")

        article_dates = art.get("ArticleDate", [])

        electronic_year = ""

        for date in article_dates:

            if date.attributes.get("DateType") == "Electronic":
                electronic_year = date.get("Year", "")
                break

        missing_date_results.append({
            "pmid": pmid,
            "journal_year": journal_year,
            "electronic_year": electronic_year
        })

        print(
            pmid,
            "→ Journal:",
            journal_year,
            "| Electronic:",
            electronic_year
        )

    except Exception as e:
        print("ERROR:", pmid, e)

    time.sleep(0.4)

NO RECORD: 41219071
NO RECORD: 41412986
NO RECORD: 41412990
NO RECORD: 41433266
NO RECORD: 41433272
NO RECORD: 41433318
NO RECORD: 41433361
NO RECORD: 41433371
NO RECORD: 41433458
NO RECORD: 41433513
NO RECORD: 41433740
NO RECORD: 41433764
NO RECORD: 41434179
NO RECORD: 41468071
NO RECORD: 41468091


In [29]:
missing_date_check = pd.DataFrame(missing_date_results)

print("Successfully retrieved:", len(missing_date_check))
print("Still missing:", len(missing_pmids) - len(missing_date_check))

Successfully retrieved: 0
Still missing: 15


In [30]:
missing_date_check

""


In [31]:
missing_15_info = df[
    df["pmid"].astype(str).isin(missing_pmids)
][
    ["pmid", "title", "year", "journal", "layer"]
].sort_values("pmid")

missing_15_info

,pmid,title,year,journal,layer
5935,41219071,Immunotherapy in advanced endometrial cancer w...,2025.0,Farmacia hospitalaria : organo oficial de expr...,L1_scientific_engagement
1406,41412986,P05 The efficacy and safety of isotretinoin in...,2025.0,The British journal of dermatology,L1_scientific_engagement
10117,41412990,O04 Mission EB -mesenchymal intravenous stroma...,2025.0,The British journal of dermatology,L2_safety_risk
16234,41433266,Technology and Dementia Preconference.,2025.0,Alzheimer's & dementia : the journal of the Al...,L3_comparative_critical
16233,41433272,Technology and Dementia Preconference.,2025.0,Alzheimer's & dementia : the journal of the Al...,L3_comparative_critical
16232,41433318,Technology and Dementia Preconference.,2025.0,Alzheimer's & dementia : the journal of the Al...,L3_comparative_critical
16231,41433361,Technology and Dementia Preconference.,2025.0,Alzheimer's & dementia : the journal of the Al...,L3_comparative_critical
16230,41433371,Technology and Dementia Preconference.,2025.0,Alzheimer's & dementia : the journal of the Al...,L3_comparative_critical
989,41433458,Alzheimer's Imaging Consortium.,2025.0,Alzheimer's & dementia : the journal of the Al...,L1_scientific_engagement
16228,41433513,Alzheimer's Imaging Consortium.,2025.0,Alzheimer's & dementia : the journal of the Al...,L3_comparative_critical


In [32]:
missing_15_info["year"].value_counts(dropna=False)

year
2025.0    15
Name: count, dtype: int64

In [33]:
pd.crosstab(
    date_check_all["journal_year"],
    date_check_all["electronic_year"]
)

electronic_year,,2024,2025,2026
journal_year,,,,
,8,0,32,0
2024,0,10,3,0
2025,2473,19,13875,5
2026,6,1,4708,101


In [34]:
date_check_all[
    ["journal_year", "electronic_year"]
].isna().sum()

journal_year       0
electronic_year    0
dtype: int64

In [35]:
date_validation = df[
    ["pmid", "year"]
].copy()

date_validation["pmid"] = (
    date_validation["pmid"].astype(str)
)

date_validation = date_validation.merge(
    date_check_all,
    on="pmid",
    how="left"
)

print("Shape:", date_validation.shape)

Shape: (21256, 13)


In [36]:
date_validation[
    date_validation["year"] !=
    pd.to_numeric(
        date_validation["journal_year"],
        errors="coerce"
    )
][
    [
        "pmid",
        "year",
        "journal_year",
        "electronic_year",
        "history_epublish_year",
        "history_aheadofprint_year",
        "history_ppublish_year"
    ]
].head(20)

,pmid,year,journal_year,electronic_year,history_epublish_year,history_aheadofprint_year,history_ppublish_year
0,41483736,2025.0,2026,2025,,,
1,41483728,2025.0,2026,2025,,,
2,41483727,2025.0,2026,2025,,,
3,41483666,2025.0,2026,2025,,,
4,41483580,2025.0,2026,2025,,,
5,41483505,2025.0,2026,2025,,,
6,41483501,2025.0,2026,2025,,,
7,41483491,2025.0,2026,2025,,,
8,41482263,2025.0,2026,2025,,,
9,41482234,2025.0,2026,2025,,,


In [37]:
pd.crosstab(
    [
        date_validation["year"],
        date_validation["journal_year"]
    ],
    date_validation["electronic_year"]
)

electronic_year            2024   2025  2026
year   journal_year                         
2024.0 2024             0    10      3     0
       2026             0     1      0     0
2025.0                  0     0     32     0
       2025          2473    19  13875     5
       2026             6     0   4708   101

In [38]:
date_validation[
    (
        date_validation["electronic_year"].astype(str) == "2026"
    )
][
    [
        "pmid",
        "year",
        "journal_year",
        "electronic_year",
        "history_epublish_year",
        "history_aheadofprint_year",
        "history_ppublish_year"
    ]
].head(30)

,pmid,year,journal_year,electronic_year,history_epublish_year,history_aheadofprint_year,history_ppublish_year
20,41481110,2025.0,2025,2026,,,
81,41478627,2025.0,2025,2026,,,
143,41475978,2025.0,2026,2026,,,
168,41474600,2025.0,2026,2026,,,
172,41474293,2025.0,2026,2026,,,
207,41472521,2025.0,2026,2026,,,
267,41469030,2025.0,2026,2026,,,
280,41468178,2025.0,2026,2026,,,
510,41459919,2025.0,2026,2026,,,
861,41441872,2025.0,2026,2026,,,


In [39]:
print(
    "Records with electronic year 2026:",
    (
        date_validation["electronic_year"].astype(str) == "2026"
    ).sum()
)

print(
    "Records with journal year 2026:",
    (
        date_validation["journal_year"].astype(str) == "2026"
    ).sum()
)

print(
    "Records with dataset year 2026:",
    (
        date_validation["year"] == 2026
    ).sum()
)

Records with electronic year 2026: 106
Records with journal year 2026: 4816
Records with dataset year 2026: 0


In [40]:
from Bio import Entrez

Entrez.email = "YOUR_EMAIL"

test_pmids = [
    "41481110",  # dataset 2025, journal 2025, electronic 2026
    "41475978",  # dataset 2025, journal 2026, electronic 2026
    "41478627",  # dataset 2025, journal 2025, electronic 2026
]

for pmid in test_pmids:

    handle = Entrez.efetch(
        db="pubmed",
        id=pmid,
        retmode="xml"
    )

    record = Entrez.read(handle)

    article = record["PubmedArticle"][0]
    medline = article["MedlineCitation"]
    pubmed_article = medline["Article"]

    print("=" * 80)
    print("PMID:", pmid)

    print("\nJOURNAL PUB DATE:")
    print(
        pubmed_article["Journal"]["JournalIssue"]["PubDate"]
    )

    print("\nARTICLE DATE:")
    print(
        pubmed_article.get("ArticleDate")
    )

    print("\nPUBMED HISTORY:")
    print(
        article.get("PubmedData", {}).get("History")
    )

PMID: 41481110

JOURNAL PUB DATE:
{'Year': '2025', 'Month': 'Dec'}

ARTICLE DATE:
[DictElement({'Year': '2026', 'Month': '01', 'Day': '02'}, attributes={'DateType': 'Electronic'})]

PUBMED HISTORY:
[DictElement({'Year': '2026', 'Month': '1', 'Day': '2', 'Hour': '13', 'Minute': '11'}, attributes={'PubStatus': 'medline'}), DictElement({'Year': '2026', 'Month': '1', 'Day': '2', 'Hour': '13', 'Minute': '10'}, attributes={'PubStatus': 'pubmed'}), DictElement({'Year': '2026', 'Month': '1', 'Day': '2', 'Hour': '9', 'Minute': '23'}, attributes={'PubStatus': 'entrez'}), DictElement({'Year': '2026', 'Month': '1', 'Day': '2'}, attributes={'PubStatus': 'pmc-release'})]
PMID: 41475978

JOURNAL PUB DATE:
{'Year': '2026', 'Month': 'Mar', 'Day': '06'}

ARTICLE DATE:
[DictElement({'Year': '2026', 'Month': '03', 'Day': '06'}, attributes={'DateType': 'Electronic'})]

PUBMED HISTORY:
[DictElement({'Year': '2025', 'Month': '10', 'Day': '31'}, attributes={'PubStatus': 'received'}), DictElement({'Year': '202

In [42]:
# ============================================================
# FINALIZE THE THESIS DATASET
# ============================================================

FINAL_COLUMNS = [
    "pmid",
    "title",
    "abstract",
    "year",
    "journal",
    "layer"
]

# Keep only the six approved thesis attributes
df_final = df[FINAL_COLUMNS].copy()

# Ensure PMID is treated as an identifier
df_final["pmid"] = df_final["pmid"].astype(str)

# Keep year numeric while preserving missing values
df_final["year"] = pd.to_numeric(
    df_final["year"],
    errors="coerce"
).astype("Int64")

# Final validation
print("FINAL DATASET SHAPE:", df_final.shape)

print("\nFINAL COLUMNS:")
print(df_final.columns.tolist())

print("\nMISSING VALUES:")
print(df_final.isnull().sum())

print("\nDUPLICATE PMIDs:")
print(df_final["pmid"].duplicated().sum())

print("\nDUPLICATE ROWS:")
print(df_final.duplicated().sum())

print("\nLAYER DISTRIBUTION:")
print(df_final["layer"].value_counts())

print("\nYEAR DISTRIBUTION:")
print(df_final["year"].value_counts(dropna=False).sort_index())

# Save final dataset
FINAL_FILE = "pubmed_medical_affairs_3layer_corpus_FINAL.csv"

df_final.to_csv(
    FINAL_FILE,
    index=False
)

print("\n====================================")
print("FINAL DATASET SAVED:")
print(FINAL_FILE)
print("====================================")

FINAL DATASET SHAPE: (21256, 6)

FINAL COLUMNS:
['pmid', 'title', 'abstract', 'year', 'journal', 'layer']

MISSING VALUES:
pmid         0
title       42
abstract     0
year         8
journal      0
layer        0
dtype: int64

DUPLICATE PMIDs:
0

DUPLICATE ROWS:
0

LAYER DISTRIBUTION:
layer
L1_scientific_engagement    9599
L3_comparative_critical     5849
L2_safety_risk              5808
Name: count, dtype: int64

YEAR DISTRIBUTION:
year
2024       14
2025    21234
<NA>        8
Name: count, dtype: Int64

FINAL DATASET SAVED:
pubmed_medical_affairs_3layer_corpus_FINAL.csv
